In [1]:
import os
import itertools
import warnings
import logging
import random

import torch
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
import numpy as np

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.molmim.infer import MolMIMInference
from nemo.collections.common.tokenizers.regex_tokenizer import RegExTokenizer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
[NeMo W 2025-08-08 13:54:55 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:257: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
      def forward(
    
[NeMo W 2025-08-08 13:54:55 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:268: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
      def backward(ctx, grad_output):
    
[NeMo W 2025-08-08 13:54:55 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:328: FutureWarni

[NeMo I 2025-08-08 13:55:08 megatron_hiddens:110] Registered hidden transform sampled_var_cond_gaussian at bionemo.model.core.hiddens_support.SampledVarGaussianHiddenTransform
[NeMo I 2025-08-08 13:55:08 megatron_hiddens:110] Registered hidden transform interp_var_cond_gaussian at bionemo.model.core.hiddens_support.InterpVarGaussianHiddenTransform


In [2]:
train_df = pd.read_csv("/workspace/bionemo/data/train_substrate.csv")
test_df = pd.read_csv("/workspace/bionemo/data/test_substrate.csv")

train_df.shape, test_df.shape

((82, 1), (21, 1))

In [3]:
max_token_length = 126
# Note: the maximum token length generated from the smiles string should be 2 less than the max_seq_length specified in the model config.
# This is to account for the extra tokens <BOS> and <EOS>

def vocab_compliance_check(smiles: str, tokenizer: RegExTokenizer, max_token_length: int) -> bool:
    """Checks if the SMILES string only contains vocabulary in the tokenizer's vocabulary
    and if the token length is less than or equal to `max_token_length"""
    tokens = tokenizer.text_to_tokens(smiles)
    vocab_allowed = tokenizer.vocab.keys()
    return set(tokens).issubset(set(vocab_allowed)) and len(tokens) <= max_token_length

model_name = "molmim"
print(f"Filtering out molecules which are not present in the {model_name} tokenizer vocabulary or with max token length greater than {max_token_length}...")
tokenizer_path = bionemo_home + "/tokenizers/molecule/{model_name}/vocab/{model_name}.{extension}"
tokenizer = RegExTokenizer().load_tokenizer(regex_file=tokenizer_path.format(model_name=model_name, extension="model"), vocab_file=tokenizer_path.format(model_name=model_name, extension="vocab"))

train_df["vocab_compliant"] = train_df["canon_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
test_df["vocab_compliant"] = test_df["canon_smiles"].apply(lambda smi: vocab_compliance_check(smi, tokenizer, max_token_length))
# Select only molecules which are vocab compliant
train_df = train_df.loc[train_df['vocab_compliant']]
test_df = test_df.loc[test_df['vocab_compliant']]

print(f"{len(train_df)} molecules in train set after filtering.")
print(f"{len(test_df)} molecules in test set after filtering.")

Filtering out molecules which are not present in the molmim tokenizer vocabulary or with max token length greater than 126...
72 molecules in train set after filtering.
18 molecules in test set after filtering.


In [6]:
task = "finetuning"  # Specify your task name

os.makedirs(f"/workspace/bionemo/data/processed/{task}/train", exist_ok=True)
os.makedirs(f"/workspace/bionemo/data/processed/{task}/val", exist_ok=True)
os.makedirs(f"/workspace/bionemo/data/processed/{task}/test", exist_ok=True)

train_df.to_csv(f"/workspace/bionemo/data/processed/{task}/train/train.csv", index=False)
test_df.to_csv(f"/workspace/bionemo/data/processed/{task}/val/val.csv", index=False)

# Create a fake test df
pd.DataFrame({"canon_smiles": ["C1=CC=CC=C1", "C1=CC=CC=C1O"]}).to_csv(f"/workspace/bionemo/data/processed/{task}/test/test.csv", index=False)

In [9]:
import subprocess

# --- Customizable Parameters ---
# Model and Data Paths
model_path = f"{bionemo_home}/models/molecule/molmim/molmim_70m_24_3.nemo"
index_mapping_dir = "data/data_index/"
data_col = 0
config_name: str = "pretrain_small_canonicalized_logv"

# Dataset Configuration
train_set = "train"
val_set = "val"
test_set = "test"

# Training Parameters
do_training = True
devices = 1
accelerator = 'gpu'
max_steps = 200
val_check_interval = 100
global_batch_size = "null"  # Use "null" as a string to be correctly interpreted in the command

# Experiment Management
create_wandb_logger = True
resume_if_exists = False

# --- Execution ---
# Clean up previous data index
if os.path.exists(index_mapping_dir):
    subprocess.run(["rm", "-rf", index_mapping_dir], check=True)


# Construct the pre-training command using an f-string
command = (
    f"python {bionemo_home}/examples/molecule/molmim/pretrain.py "
    f"--config-path=/workspace/bionemo/examples/molecule/molmim/conf "
    f"--config-name={config_name} "
    f"restore_from_path={model_path} "  # <-- KEY: Load the vanilla model
    f"do_training={do_training} "
    f"++model.data.dataset_path=data/processed/{task}/ "
    f"++model.data.dataset.train={train_set} "
    f"++model.data.dataset.val={val_set} "
    f"++model.data.dataset.test={test_set} "
    f"++model.data.index_mapping_dir={index_mapping_dir} "
    f"++model.data.data_impl_kwargs.csv_mmap.data_col={data_col} "
    f"++model.dwnstr_task_validation.enabled=False "
    f"++model.global_batch_size={global_batch_size} "
    f"++trainer.devices={devices} "
    f"++trainer.accelerator='{accelerator}' "
    f"++trainer.max_steps={max_steps} "
    f"++trainer.val_check_interval={val_check_interval} "
    f"++exp_manager.create_wandb_logger={create_wandb_logger} "
    f"++exp_manager.resume_if_exists={resume_if_exists} "
)

print(command)

python /workspace/bionemo/examples/molecule/molmim/pretrain.py --config-path=/workspace/bionemo/examples/molecule/molmim/conf --config-name=pretrain_small_canonicalized_logv restore_from_path=/workspace/bionemo/models/molecule/molmim/molmim_70m_24_3.nemo do_training=True ++model.data.dataset_path=data/processed/finetuning/ ++model.data.dataset.train=train ++model.data.dataset.val=val ++model.data.dataset.test=test ++model.data.index_mapping_dir=data/data_index/ ++model.data.data_impl_kwargs.csv_mmap.data_col=0 ++model.dwnstr_task_validation.enabled=False ++model.global_batch_size=null ++trainer.devices=1 ++trainer.accelerator='gpu' ++trainer.max_steps=200 ++trainer.val_check_interval=100 ++exp_manager.create_wandb_logger=True ++exp_manager.resume_if_exists=False 


In [ ]:
# Copy the latest trained model to the model directory
base_path = "/result/nemo_experiments/MolMIM/"
checkpoint_path = os.path.join(base_path, f"MolMIM-{config_name.split('_')[1]}_finetuning", "checkpoints", "MolMIM.nemo")
print(f"Checkpoint path: {checkpoint_path}, is file: {os.path.isfile(checkpoint_path)}")

os.makedirs(os.path.join(bionemo_home, "data", "models"), exist_ok=True)

os.system(f"mv {checkpoint_path} {bionemo_home}/data/models/MolMIM_{config_name.split('_')[1]}_{task}_max_steps_{max_steps}.nemo")